# 2장. CSV 분석 에이전트 실행

메시지 CSV를 보내고 분석 JSON을 받습니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | Session 생성·스트리밍·완료 상태와 응답 확인 |
| 이번 실습 조합 | OpenAI-hosted · 표준 라이브러리 · 메시지 입력 |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

1장의 Agent·Session·Turn·Environment 구분을 실제 요청과 연결합니다. 빠른 시작을 실행했다면 같은 API 흐름에 CSV 분석 요청을 적용한다고 생각하세요.

| 단계 | 직접 사용하는 API 또는 값 | 확인할 내용 |
| --- | --- | --- |
| 요청 구성 | Agent 지침·환경·입력 메시지 | 누가, 어디에서, 무엇을 수행하는지 |
| 실행 | `sessions.create()`, `stream=True` | 세션 생성 이벤트와 텍스트 출력 |
| 결과 확인 | `get_final_result()`, `turns.retrieve()` | 완료 상태와 분석 JSON |
| 정리 | 정리 helper의 삭제·재조회 | 생성한 세션의 API상 부재 |

이 장은 파일 업로드 없이 작은 CSV를 메시지에 넣고 **텍스트 JSON**을 받습니다. 5장에서는 파일 업로드와 보고서 다운로드를 실습합니다.

수동 실습은 아래 셀을 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_intro.py --chapter 2 --live`로 실행합니다.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실습 준비와 실행 모드

로컬에서는 저장소 루트에서 `uv sync --locked`, `uv run jupyter lab`을 실행하고 `.venv` 커널을 선택하세요.
Colab에서는 다운로드 안내의 독립 실행본을 업로드하세요. 준비 셀이 패키지를 직접 설치하고 포함된 코드·데이터를 생성합니다.
실제 API 실습은 **RUN_LIVE=True**로 명시적으로 허용합니다.
키는 `.env.local` 또는 Colab Secrets에서 읽으며 출력하지 않습니다. 라이브 실행에는 비용이 발생합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 2
CHAPTER = "02_first_agent"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print(
    {
        "chapter": NUMBER,
        "live_enabled": settings.enabled,
        "model": settings.model or "Offline mode",
    }
)


## 1. 입력 CSV 확인하기

각 행은 날짜·상품·수량·단가를 가진 거래입니다. 에이전트에게 원문과 계산 규칙을 보내고, 미리 계산한 정답은 보내지 않습니다. 아래 출력은 로컬 파일 확인이며 API 호출이나 분석 실행은 아닙니다.


In [ ]:
csv_text = (settings.root / "data/sales.csv").read_text()
print(csv_text)


## 2. Agent 지침·환경·입력 구성하기

- Agent 지침은 Python으로 계산하고 입력에 없는 거래를 만들지 않도록 정합니다.
- 입력 메시지는 CSV 저장·분석 작업과 최종 JSON 형식을 지정합니다. `METRICS_CONTRACT`를 조합해 결과 형식을 한 곳에서 관리합니다.
- 환경은 `openai_hosted` sandbox입니다. `network.access="disabled"`는 sandbox의 외부 네트워크 접근을 끕니다. 로컬 클라이언트의 OpenAI API 연결 설정과는 별개입니다.

이 예제는 표준 라이브러리만 사용하므로 패키지 설치가 필요 없습니다. `agent` 설정을 세션 생성 요청에 직접 넣으며, 별도의 저장 Agent를 생성하지 않습니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)
task = (
    "Save the CSV below as /workspace/sales.csv. "
    "Write and run a Python standard-library analysis script, inspect stdout, "
    "and calculate the summary. "
    + METRICS_CONTRACT
    + "\nCSV input:\n"
    + csv_text
)


In [ ]:
environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "network": {"access": "disabled"},
}
print({"environment": environment, "input_characters": len(task)})


## 3. 세션 생성과 응답 스트리밍

아래 API 준비 셀부터 실제 요청이 가능합니다. 실행 셀과 다음 결과 확인 셀은 오류·일반적인 인터럽트에서 `cleanup_on_error()`로 정리를 시도합니다. 정상 완료 후에는 **5절의 정리 셀까지 실행**하세요. 중간에 학습을 멈추더라도 정리 셀을 실행해야 합니다.

스트림 컨텍스트는 한 셀에서 열고 닫습니다. 스트림이 닫혀도 세션은 남아 있으므로 결과 확인과 세션 삭제를 따로 배웁니다. 커널·프로세스 강제 종료에서는 정리가 실행되지 않을 수 있습니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

`output_dir`는 보고서 다운로드 폴더가 아니라 `creation.json` 등 복구용 기록을 저장할 로컬 경로입니다. 고유 metadata와 이벤트에서 받은 Session ID로 이번 실행의 세션만 추적합니다. 상태 준비 셀은 실행을 시작할 때 한 번만 실행하세요. 결과 확인·정리 중 다시 실행하면 자원 식별자를 잃을 수 있습니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/intro" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = actual = None


### 스트림에서 관찰할 내용

| 이벤트 또는 처리 | 이 셀의 행동 |
| --- | --- |
| `agent.session.created` | Session ID를 즉시 저장하고 표시 |
| `agent.session.turn.output_text.delta` | 도착하는 텍스트를 표시 |
| `agent.session.requires_action` | 이 실습에서 예상하지 않은 외부 작업 요청을 오류로 처리 |
| `check_event()` | 나머지 이벤트에서 세션·환경·root Turn 실패나 취소 감지 |

`with_result_collection()`은 이벤트를 관찰하는 동안 최종 응답을 수집합니다. 루프가 끝나면 `get_final_result()`로 결과를 꺼냅니다. 이벤트의 개수·순서는 고정하지 않습니다. `idle`이나 연결 종료만으로 작업 성공을 판단하지 않습니다.

타이머는 API 실행 직전에 시작합니다. 600초 제한은 **이벤트를 받을 때 확인**하며, 무응답 중 강제 중단하는 전체 타이머는 아닙니다. 클라이언트 HTTP timeout은 90초입니다. 중단하려면 Jupyter 인터럽트를 사용하고 정리 결과를 확인하세요.

다음 셀은 작업을 제출합니다. 재실행으로 중복 작업을 만들지 않도록 같은 실행 상태에서 두 번째 제출은 거절합니다. 출력된 텍스트가 보여도 아직 계산값 검증까지 끝난 것은 아닙니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit the task.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
    sessions.setdefault(result.session_id, {"deleted": False})
    print("\nResponse collected. Inspect the completed turn in the next cell.")


## 4. 완료 상태와 분석 결과 확인하기

스트리밍으로 수집한 `result.output_text`를 JSON으로 읽고, `turns.retrieve()`로 저장된 root Turn도 조회합니다. 이 셀은 새 작업을 제출하지 않습니다.

| 확인 항목 | 알 수 있는 것 |
| --- | --- |
| 수집 결과와 조회한 Turn의 `completed` | 작업이 완료 상태로 끝났는지 |
| 출력한 분석 JSON | 에이전트가 반환한 행 수·수량·매출 |
| 별도 검증기의 명령·정답 대조 | Python 실행 성공과 계산 정확성 |

`json.loads()` 성공은 JSON 문법 확인이며 필드 형식이나 계산 정확성을 보장하지 않습니다. `get_final_result()`도 실패·취소를 성공으로 바꾸지 않습니다. 아래 표시와 별도 검증 도구의 통과 기록을 구분하세요.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if result is None:
        raise RuntimeError("Run the session creation and streaming cell first.")
    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15,
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    actual = json.loads(result.output_text)
    print("Session ID:", result.session_id)
    print("Turn ID:", result.turn_id)
    print("Turn status:", turn.status)
    print(json.dumps(actual, indent=2))


## 5. 세션 삭제와 정리 확인하기

이제 필요한 결과를 로컬 변수에 보관했으므로 세션을 정리합니다. 아래 셀은 분석을 다시 실행하지 않습니다. 중간에 학습을 멈추는 경우에도 실행하세요.

`cleanup_resources()`는 이번 실행의 고유 metadata로 생성 응답을 놓친 세션도 찾고, 실행 중이면 취소한 뒤 삭제를 시도합니다. 삭제 충돌은 제한적으로 재시도하고 재조회 `404`로 API상 부재를 확인합니다. 물리적 sandbox 정리가 동기적으로 완료되었다는 의미는 아닙니다. 이 장은 별도 저장 Agent를 만들지 않으므로 Agent 목록은 비어 있습니다.

출력의 `deleted`와 `verified_absent`가 모두 참인지 확인하세요. 정리 중 오류가 있으면 성공 문구 대신 오류가 발생합니다. 검증 도구에서는 삭제 전에 검사를 추가하며, 검사 오류가 발생해도 정리를 시도하고 원래 오류를 보존합니다.

결과 객체와 `actual`은 삭제 후에도 로컬에서 읽을 수 있습니다. 추가 서버 조회에는 삭제한 세션을 사용할 수 없습니다.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 6. 기대 결과·실패 시 조치·다음 장

수동 실습에서는 출력 JSON의 행 수 6, 판매 수량 11, 총매출 127500, 상품별 매출 A=30000·B=75000·C=22500을 확인하세요. 이 값은 독자의 비교 기준이며 API 입력에는 포함하지 않습니다. 별도 검증기는 실제 명령 종료 코드와 계산값을 함께 대조합니다.

| 상황 | 할 일 |
| --- | --- |
| `RUN_LIVE=False` 준비 오류 | 실제 호출하려면 설정 셀을 수정하고 다시 실행. 오프라인 검증은 실행 스크립트 사용 |
| 401/403 | 인증과 프로젝트 접근 권한 확인 |
| 429 | API 제한 확인 |
| 연결 종료·실행 중단 | 정리 결과 확인. 실패가 남으면 저장된 ID와 상태를 확인한 뒤 복구 |
| 커널·프로세스 강제 종료 | `outputs/intro/<chapter>/<run_id>/creation.json`과 Session ID로 이번 실행만 식별 |

키나 원본 오류 body를 출력하지 마세요. 연결이 끊겼다고 같은 입력을 새 세션에 무조건 재전송하지 않습니다. 복구 절차는 [빠른 시작](../quickstart.ipynb)의 실패·중단 후 다음 행동를 참고하세요. 검증 도구로 실행했다면 로컬 `summary.json`도 확인할 수 있습니다.

이 장의 세션은 정리했으므로 3장에서 재사용하지 않습니다. **[3장](03_sessions.ipynb)은 새 세션을 만들고 그 안에서 후속 입력을 보내는 방식**으로 일별 매출 분석을 이어갑니다.


## 공식 참고 자료

### Agents API 전용 문서

- [Agents API quickstart](https://developers.openai.com/api/docs/guides/agents-api/quickstart) — 세션 생성·스트리밍·결과 확인·자원 정리.
- [Events and items](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) — 이벤트 스트림·완료 상태·저장된 items와 turns 조회.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
